In [25]:
import sys
import warnings
import numpy as np
import pathlib as pl
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm
from scipy.optimize import minimize
root_folder = pl.Path.cwd().parents[2]
sys.path.insert(0, str(root_folder / "utilities"))
import common_functions as cf

warnings.filterwarnings('ignore')

initial_data_folder = "data/initial_data/function_7"
initial_inputs_path = pl.Path.joinpath(root_folder, initial_data_folder,  "initial_inputs.npy")
initial_outputs_path = pl.Path.joinpath(root_folder, initial_data_folder, "initial_outputs.npy")

In [26]:
data_in = np.load(initial_inputs_path)
data_out = np.load(initial_outputs_path)

Week-01

In [27]:
X_init = data_in
y_init = data_out

y_best = y_init.max()  # best performance so far

kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_init, y_init)

# --- Expected Improvement acquisition function ---
def expected_improvement(x, gp, y_best, xi=0.01):
    x = np.array(x).reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    mu, sigma = mu[0], sigma[0]
    if sigma == 0.0:
        return 0.0
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -ei  # negative for minimization in scipy

# --- Optimize acquisition function ---
bounds = [(0,1)]*6
best_x = None
best_ei = float('inf')

for _ in range(20):  # multiple random starts
    x0 = np.random.rand(6)
    res = minimize(lambda x: expected_improvement(x, gp, y_best),
                   x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_ei:
        best_ei = res.fun
        best_x = res.x

x_next = best_x
print("Next hyperparameter set to try:", x_next)

Next hyperparameter set to try: [0.28868963 0.79883816 0.41153751 0.77603141 0.35972398 0.64797378]


Week-02

In [28]:
#x_new = np.array([[0.000000,0.210370,1.000000,0.000000,0.323749,1.000000]])
#y_new = np.array([0.543430844426678])

#X_init = np.vstack([X_init, x_new])
#y_init = np.hstack([y_init, y_new])

X_new = np.array([
    [0.000000,0.210370,1.000000,0.000000,0.323749,1.000000]
])

y_new = np.array([ 
    0.543430844426678
])

# Combine all data
X_init = np.vstack((data_in, X_new))
y_init = np.concatenate((data_out, y_new))


y_best = y_init.max()  # best performance so far

In [29]:
# --- Fit Gaussian Process ---
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_init, y_init)
y_best = y_init.max()  # still the best observed performance

In [30]:
def expected_improvement(x, gp, y_best, xi=0.05):
    x = np.array(x).reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    mu, sigma = mu[0], sigma[0]
    if sigma == 0:
        return 0
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -ei  # negative for minimize

In [31]:
bounds = [(0, 1)] * 6
best_x = None
best_ei = float('inf')

for _ in range(20):
    x0 = np.random.rand(6)
    res = minimize(lambda x: expected_improvement(x, gp, y_best),
                   x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_ei:
        best_ei = res.fun
        best_x = res.x

x_next = best_x
print("Next hyperparameter set to try:", x_next)

Next hyperparameter set to try: [0.88776994 0.7012333  0.65755415 0.56059676 0.08233449 0.0074044 ]


Week-03

In [32]:
X_new = np.array([
    [0.000000,0.210370,1.000000,0.000000,0.323749,1.000000],
    [0.403152,0.365261,0.889930,0.311536,0.081832,0.228459]
])

y_new = np.array([ 
    0.543430844426678,
    0.2524752216448303
])

# Combine all data
X_init = np.vstack((data_in, X_new))
y_init = np.concatenate((data_out, y_new))


# --- Update best observed performance ---
y_best = y_init.max()

# --- Fit Gaussian Process ---
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_init, y_init)

# --- Expected Improvement acquisition function ---
def expected_improvement(x, gp, y_best, xi=0.05):  # increase xi for exploration
    x = np.array(x).reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    mu, sigma = mu[0], sigma[0]
    if sigma == 0.0:
        return 0.0
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -ei  # negative for minimization

# --- Optimize acquisition function ---
bounds = [(0, 1)] * 6
best_x = None
best_ei = float('inf')

# Use more random restarts for robust optimization
n_restarts = 50
for _ in range(n_restarts):
    x0 = np.random.rand(6)
    res = minimize(lambda x: expected_improvement(x, gp, y_best, xi=0.05),
                   x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_ei:
        best_ei = res.fun
        best_x = res.x

x_next = best_x
print("Next hyperparameter set to try:", x_next)

res_formatted = [f"{r:.6f}" for r in x_next]
result = "-".join(res_formatted)
print(result)

Next hyperparameter set to try: [0.74343335 0.43259054 0.83825522 0.09982933 0.2090132  0.60766169]
0.743433-0.432591-0.838255-0.099829-0.209013-0.607662
